# Scraping Boursorama - Cotation CAC40 (3 mois)

**Objectif :** Télécharger automatiquement les données de cotation du CAC40 sur 3 mois depuis Boursorama.

In [20]:
!pip install selenium webdriver-manager pandas

In [21]:
import time
import random
import os
import glob
import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

## 1. Configurer Chrome pour télécharger automatiquement

On indique à Chrome où sauvegarder le fichier téléchargé, sans qu'il demande confirmation.

In [22]:
# Dossier où le fichier sera téléchargé 
dossier_telechargement = os.getcwd()
print(f"Les fichiers seront téléchargés dans : {dossier_telechargement}")

# On configure Chrome
options = webdriver.ChromeOptions()

# On dit à Chrome de télécharger sans demander où sauvegarder
options.add_experimental_option('prefs', {
    'download.default_directory': dossier_telechargement,  # dossier de destination
    'download.prompt_for_download': False,                 # pas de popup "où sauvegarder"
    'download.directory_upgrade': True,
    'safebrowsing.enabled': True                           # évite les blocages de sécurité
})

# On lance Chrome avec ces options
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

# On ouvre directement la page du CAC40 sur Boursorama
driver.get("https://www.boursorama.com/bourse/indices/cours/1rPCAC/")
print("Page ouverte.")

# On attend et on accepte la bannière de cookies
try:
    cookies_btn = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.ID, "didomi-notice-agree-button"))
    )
    time.sleep(round(random.uniform(1, 2), 1))
    cookies_btn.click()
    print("Cookies acceptés.")
except:
    # Si pas de bannière cookies, on continue quand même
    print("Pas de bannière cookies détectée, on continue.")

# On attend que la page se charge
time.sleep(round(random.uniform(3, 5), 1))

Les fichiers seront téléchargés dans : c:\Users\ilham\Documents\Perso\Ilham\IPSSI\IPPSI_scrapping\TP SOLO
Page ouverte.
Cookies acceptés.


## 3. Cliquer sur le bouton "3M" pour afficher 3 mois de données

In [23]:
# On cherche le bouton 3M grâce à son attribut data-brs-quote-chart-duration-length="90"
btn_3m = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable(
        (By.CSS_SELECTOR, "div[data-brs-quote-chart-duration-length='90']")
    )
)

# On scrolle jusqu'au bouton pour qu'il soit visible
driver.execute_script("arguments[0].scrollIntoView(true);", btn_3m)

# On attend un peu plus longtemps pour laisser les animations/bannières se terminer
time.sleep(round(random.uniform(2, 4), 1))

# On clique via JavaScript pour éviter qu'un élément superposé bloque le clic
driver.execute_script("arguments[0].click();", btn_3m)
print("Bouton 3M cliqué.")

# On laisse le graphique se mettre à jour
time.sleep(round(random.uniform(3, 5), 1))

Bouton 3M cliqué.


## 4. Cliquer sur le bouton de téléchargement

In [24]:
# On cherche le bouton de téléchargement via l'icône avec la classe c-icon--download
btn_download = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable(
        (By.CSS_SELECTOR, "span.c-icon--download")
    )
)

# On scrolle jusqu'au bouton
driver.execute_script("arguments[0].scrollIntoView(true);", btn_download)
time.sleep(round(random.uniform(2, 4), 1))

# On clique via JavaScript pour éviter qu'un élément superposé bloque le clic
driver.execute_script("arguments[0].click();", btn_download)
print("Bouton de téléchargement cliqué.")

# On attend que le fichier soit téléchargé
time.sleep(5)
print("Téléchargement terminé.")

Bouton de téléchargement cliqué.
Téléchargement terminé.


## 5. Lire le fichier téléchargé

In [25]:
# On cherche le fichier le plus récent téléchargé dans le dossier
# Boursorama télécharge un fichier .txt
fichiers = glob.glob(os.path.join(dossier_telechargement, "*.txt"))

if not fichiers:
    print("Aucun fichier .txt trouvé. Vérifie le dossier de téléchargement.")
else:
    # On prend le fichier le plus récent
    fichier_cac = max(fichiers, key=os.path.getctime)
    print(f"Fichier trouvé : {fichier_cac}")

    # On ouvre le fichier pour voir son contenu brut
    with open(fichier_cac, 'r', encoding='utf-8') as f:
        contenu = f.read()
    print("\nAperçu du contenu brut :")
    print(contenu[:500])

Fichier trouvé : c:\Users\ilham\Documents\Perso\Ilham\IPSSI\IPPSI_scrapping\TP SOLO\CAC40_2026-05-04.txt

Aperçu du contenu brut :
date	ouv	haut	bas	clot	vol	devise	
04/02/2026 00:00	8219.96	8298.41	8203.27	8262.16	4973	Pts	
05/02/2026 00:00	8288.47	8312.68	8193.2	8238.17	4471	Pts	
06/02/2026 00:00	8214.85	8287.92	8175.09	8273.84	4447	Pts	
09/02/2026 00:00	8294.83	8323.28	8258.75	8323.28	3468	Pts	
10/02/2026 00:00	8348.82	8369.71	8316.61	8327.88	4050	Pts	
11/02/2026 00:00	8344.25	8358.23	8277.96	8313.24	4640	Pts	
12/02/2026 00:00	8432.15	8437.35	8332.73	8340.56	5155	Pts	
13/02/2026 00:00	8315.5	8331.96	8275.04	8311.74	5306	


## 6. Mettre les données dans un DataFrame pandas

In [26]:
# On lit le fichier avec pandas
# Le fichier Boursorama est séparé par des point-virgules
df_cac = pd.read_csv(fichier_cac, sep=';', encoding='utf-8')

# Affichage du tableau
print(f"Nombre de lignes : {len(df_cac)}")
df_cac

Nombre de lignes : 61


,date\touv\thaut\tbas\tclot\tvol\tdevise\t
0,04/02/2026 00:00\t8219.96\t8298.41\t8203.27\t8...
1,05/02/2026 00:00\t8288.47\t8312.68\t8193.2\t82...
2,06/02/2026 00:00\t8214.85\t8287.92\t8175.09\t8...
3,09/02/2026 00:00\t8294.83\t8323.28\t8258.75\t8...
4,10/02/2026 00:00\t8348.82\t8369.71\t8316.61\t8...
...,...
56,27/04/2026 00:00\t8160.18\t8214.63\t8127.83\t8...
57,28/04/2026 00:00\t8129.35\t8172.89\t8090.7\t81...
58,29/04/2026 00:00\t8088.98\t8110.62\t8036.46\t8...
59,30/04/2026 00:00\t7962.84\t8115.08\t7957.83\t8...


In [27]:
output_file = "quotation.csv"
df_cac.to_csv(output_file, index=False)
print(f"Fichier exporte : {output_file}")

Fichier exporte : quotation.csv


In [28]:
# On ferme le navigateur
driver.quit()
print("Navigateur fermé.")

Navigateur fermé.
